### Importing libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import savemat
from scipy.signal import welch
import mne

output_dir = Path("../data/processed")
THETA_BAND = (4, 8)



### Making a function to extract the theta power

In [5]:
def extract_theta_power(epoch_data, band=THETA_BAND):
     """Extracts the theta power from the given epoch data using Welch's method.
     
     Parameters:
     epoch_data (epo.fif): The epoch data being extracted
     band (tuple): The frequency band to be extracted, default is theta band of (4, 8)
     
     Returns: 
          average (numpy.ndarray): The average theta power across the epoch
     """
     
     # Loading the epochs
     epochs = mne.read_epochs(epoch_data, preload=True, verbose=False)
     spectrum = epochs.compute_psd(method='welch', fmin=1, fmax=40)
     psds, freqs = spectrum.get_data(return_freqs=True) # psds shape is (n_epochs, n_channels, n_freqs)
     band_mask = (freqs >= band[0]) & (freqs <= band[1])
     # average the psd across the theta band 
     average = psds[:, :, band_mask].mean(axis=(1,2)) # shape is (n_epochs, n_channels)
     return average


### Example Theta-Power Table

In [6]:
rows = []
epoch_files = sorted(output_dir.glob("*.fif"))
print(f"Processed segment files #{len(epoch_files)}")

for file in epoch_files:
     # format reminder: subject_##_test#_phase#-epo.fif
     stem = file.stem.replace("-epo", "")
     parts = stem.split("_")
     subject = f"{parts[0]}_{parts[1]}"
     test = int(parts[2].replace("test", ""))
     phase = int(parts[3].replace("phase", ""))
     
     theta_power = extract_theta_power(file)
     for p in theta_power:
          rows.append([subject, test, phase, p])

features_df = pd.DataFrame(rows, columns=["subject", "test", "phase", "theta_power"])
features_df.to_csv(output_dir / "theta_power_features.csv", index=True)
print(f"Saved theta power features to theta_power_features.csv with shape {features_df.shape}")
features_df.groupby("test")["theta_power"].describe()


Processed segment files #143
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Effective window si

,count,mean,std,min,25%,50%,75%,max
subject,,,,,,,,
subject_01,3485.0,5.764912e-12,5.745112e-12,1.169822e-12,2.743437e-12,3.791310e-12,6.263732e-12,5.707672e-11
subject_02,2696.0,4.089498e-12,4.959329e-12,1.546807e-13,2.370621e-12,3.281263e-12,4.744303e-12,1.913335e-10
subject_03,2872.0,5.078880e-12,4.702725e-12,6.998778e-13,2.388993e-12,3.532116e-12,5.839544e-12,7.355403e-11
subject_04,2525.0,5.315817e-12,3.478545e-12,1.008199e-12,2.977767e-12,4.163556e-12,6.526338e-12,3.231801e-11
subject_05,2988.0,5.703887e-12,4.879444e-12,9.614099e-13,2.442892e-12,3.495932e-12,7.619057e-12,3.642572e-11
subject_06,2430.0,4.913845e-12,6.400892e-12,8.100352e-13,2.055001e-12,2.944477e-12,4.932874e-12,9.612657e-11
subject_07,2614.0,4.359488e-12,4.090351e-12,7.191501e-13,1.874686e-12,2.858440e-12,5.180446e-12,4.060414e-11
subject_08,2420.0,7.835627e-12,6.038630e-12,1.255416e-12,4.018803e-12,5.802611e-12,9.485354e-12,5.849094e-11
subject_09,1710.0,4.402564e-12,4.541705e-12,6.024120e-13,1.827397e-12,2.628409e-12,4.849915e-12,3.354222e-11


### Export one epoch to check against MATLAB

In [8]:
example_path = output_dir / "subject_07_test1_phase2-epo.fif"
example_epochs = mne.read_epochs(example_path, preload=True)
sfreq = example_epochs.info['sfreq']

example_signal = example_epochs.get_data()[0].mean(axis=0) # takes just the first epoch
f, pxx = welch(example_signal, fs=sfreq, nperseg=min(256, len(example_signal)))
# 256 to maximum of the signal's length in order to prevent errors

python_theta_power = np.trapezoid(pxx[(f >= THETA_BAND[0]) & (f <= THETA_BAND[1])], f[(f >= THETA_BAND[0]) & (f <= THETA_BAND[1])])

print(f"Example signal length: {len(example_signal)} is sampled at {sfreq} Hz")
print(f"Python calculated theta power (welch): {python_theta_power:.4e}")

savemat(output_dir / "matlab_check.mat", {
     "example_signal": example_signal,
     "sfreq": sfreq,
     "python_theta_power": python_theta_power
})  

print("Finished making and saving matlab_check.mat")

Reading c:\Users\tmedo\UltraDrive\Desktop\python-programs\pilot-workload-monitor\notebooks\..\data\processed\subject_07_test1_phase2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =       0.00 ...    1992.19 ms
        0 CTF compensation matrices available
Not setting metadata
300 matching events found
No baseline correction applied
0 projection items activated
Example signal length: 256 is sampled at 128.0 Hz
Python calculated theta power (welch): 1.9655e-12
Finished making and saving matlab_check.mat


### Load MATLAB results into Python

In [9]:
anova_results = pd.read_csv(output_dir / "matlab_anova_results.csv")
anova_results

,anova_p_value,crosscheck_pct_diff
0,4.756163e-75,39.916684


### Marking Results

Python theta power: 1.9655e-12
MATLAB: 2.750e-12
Rel Diff: 39.92%
p-value: 4.756163e-75

Yes it did because the ANOVA p-value was less than 0.05
which means that the difference in theta power across workload levels was statistically significant and did not occur due to chance.

Because of this, yes I find the cross-check having not landed close enough to trust the feature extraction pipeline. 

As a result, I believe I will need to use either Python or MATLAB solely for band power analysis.